# 🔎 Lab W3-1 — Grain Detective

**รายวิชาระบบสนับสนุนการตัดสินใจ · สัปดาห์ที่ 3 — Data Management & Data Warehouse**

Lab นี้ใช้คู่กับสื่อจำลอง **Grain Detective** (`/sims/grain-detective`)
ตัวเลขที่คุณคำนวณได้ในสมุดเล่มนี้ต้องตรงกับตัวเลขบนหน้าจอสื่อจำลองทุกหลัก

## สิ่งที่จะได้เรียนรู้
1. อธิบายได้ว่า **grain** ของ fact table คืออะไร และเหตุใดจึงเป็นการตัดสินใจที่ย้อนกลับยากที่สุด
2. ตรวจจับการ **นับซ้ำ (double counting)** ที่เกิดจากการเลือก grain ผิด
3. เขียน **grain contract** เป็นประโยคเดียวที่ทดสอบได้

## ข้อมูล
`pos_receipt_lines.csv` — ไฟล์ส่งออกจากระบบ POS ที่มีข้อมูล **สามระดับปนกัน**

In [ ]:
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

URL = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
       "master/datasets/week03/pos_receipt_lines.csv")
df = pd.read_csv(URL)

print(f"จำนวนแถวในไฟล์ดิบ : {len(df):,}")
print(f"จำนวนคอลัมน์      : {df.shape[1]}")
df.head(6)

## ส่วนที่ 1 — สำรวจว่าไฟล์นี้มีกี่ระดับกันแน่

ก่อนออกแบบอะไรทั้งสิ้น ต้องตอบให้ได้ก่อนว่า **หนึ่งแถวในไฟล์นี้แทนอะไร**

In [ ]:
print("จำนวนค่าที่ไม่ซ้ำในแต่ละคีย์ที่เป็นไปได้")
print(f"  receipt_no เท่านั้น              : {df.receipt_no.nunique():,}")
print(f"  receipt_no + line_no             : {df.groupby(['receipt_no','line_no']).ngroups:,}")
print(f"  receipt_no + payment_seq         : {df.groupby(['receipt_no','payment_seq']).ngroups:,}")
print(f"  receipt_no + line_no + payment_seq: {df.groupby(['receipt_no','line_no','payment_seq']).ngroups:,}")
print(f"  จำนวนแถวจริงในไฟล์               : {len(df):,}")

> **สังเกต** ตัวเลขสุดท้ายเท่ากับบรรทัดก่อนหน้า แปลว่า grain ของ *ไฟล์* คือ
> `receipt_no × line_no × payment_seq` — เป็นผลคูณไขว้ที่ระบบ POS สร้างขึ้นตอน export
>
> นี่คือ grain ที่ **ไม่มีความหมายทางธุรกิจ** เพราะไม่มีเหตุการณ์จริงใดในโลกที่เกิดขึ้น
> "หนึ่งครั้งต่อหนึ่งคู่ของรายการสินค้าและวิธีชำระเงิน"

### 🧑‍💻 งานที่ 1
คำนวณยอดขายสุทธิรวมสองแบบแล้วเปรียบเทียบ

1. `total_raw` — บวก `line_net_amount` จากไฟล์ดิบตรงๆ
2. `total_line` — บวกหลังยุบให้เหลือหนึ่งแถวต่อ `(receipt_no, line_no)`

แล้วพิมพ์ส่วนต่างออกมาเป็นเปอร์เซ็นต์

*เฉลยที่ถูกต้อง: ยอดที่ถูกคือ 23,175,222.85 บาท และไฟล์ดิบให้ตัวเลขสูงเกินจริง 12.96%*

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 2 — หาต้นตอของการนับซ้ำ

การนับซ้ำไม่ได้เกิดกับทุกใบเสร็จ ให้หาว่าเกิดกับใบเสร็จแบบไหน

### 🧑‍💻 งานที่ 2
หาว่าใบเสร็จที่ทำให้เกิดการนับซ้ำมีลักษณะอย่างไร

แนวทาง: นับจำนวน `payment_seq` ที่ไม่ซ้ำของแต่ละใบเสร็จ แล้วเทียบจำนวนแถวในไฟล์
ของใบเสร็จที่มีวิธีชำระเดียว กับใบเสร็จที่มีสองวิธี

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 3 — คำถามธุรกิจ 6 ข้อ ภายใต้ grain ต่างกัน

ตารางเดียวตอบทุกคำถามไม่ได้ — พิสูจน์ด้วยตัวเลข

In [ ]:
def collapse(data: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    """ยุบให้เหลือหนึ่งแถวต่อหนึ่งคีย์ (เลียนแบบการโหลดเข้า fact table ที่ grain นั้น)"""
    return data.drop_duplicates(keys)

CATEGORY = {
    "P-101": "เครื่องดื่ม", "P-102": "เครื่องดื่ม", "P-103": "เครื่องดื่ม",
    "P-201": "ของใช้ในบ้าน", "P-202": "ของใช้ในบ้าน", "P-203": "ของใช้ในบ้าน",
    "P-301": "เครื่องใช้ไฟฟ้า", "P-302": "เครื่องใช้ไฟฟ้า",
}

grains = {
    "ไฟล์ดิบ (line × payment)": ["receipt_no", "line_no", "payment_seq"],
    "ระดับรายการสินค้า": ["receipt_no", "line_no"],
    "ระดับใบเสร็จ": ["receipt_no"],
    "ระดับการชำระเงิน": ["receipt_no", "payment_seq"],
}
for name, keys in grains.items():
    print(f"{name:28s} → {len(collapse(df, keys)):>7,} แถว")

### 🧑‍💻 งานที่ 3
สร้างตารางเปรียบเทียบคำตอบของคำถามทั้ง 6 ข้อ ภายใต้ grain ทั้ง 4 แบบ

| # | คำถามธุรกิจ | คำตอบที่ถูกต้อง |
|---|---|---|
| 1 | ยอดขายสุทธิรวม | 23,175,222.85 บาท |
| 2 | จำนวนชิ้นของ P-101 | 3,664 ชิ้น |
| 3 | ยอดขายหมวดเครื่องใช้ไฟฟ้า | 21,827,100.50 บาท |
| 4 | มูลค่าเฉลี่ยต่อใบเสร็จ | 3,025.52 บาท |
| 5 | สัดส่วนยอดที่ชำระด้วยพร้อมเพย์ | 24.36% |
| 6 | จำนวนใบเสร็จที่ชำระมากกว่า 1 วิธี | 916 ใบ |

ทำเป็น `DataFrame` ที่มี grain เป็นแถวและคำถามเป็นคอลัมน์

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **ข้อสรุปที่ต้องเขียนลงในใบงาน**
>
> ไม่มี grain ใดตอบถูกครบทั้ง 6 ข้อ — grain ระดับรายการสินค้าตอบคำถาม Q1–Q3 ได้ถูกต้อง
> ส่วน grain ระดับการชำระเงินตอบ Q5–Q6 ได้ถูกต้อง
>
> ระบบจริงจึงต้องมี fact table **มากกว่าหนึ่งตาราง** ที่ grain ต่างกัน
> และเชื่อมกันด้วย `receipt_no` — ไม่ใช่พยายามยัดทุกอย่างลงตารางเดียว

## ส่วนที่ 4 — ออกแบบ fact table ที่ถูกต้อง

### 🧑‍💻 งานที่ 4
สร้าง fact table สองตารางจากไฟล์ดิบ

* `fact_sales_line` — grain: หนึ่งรายการสินค้าในหนึ่งใบเสร็จ
* `fact_payment` — grain: หนึ่งการชำระเงินในหนึ่งใบเสร็จ

แล้ว**พิสูจน์** ว่า
1. ผลรวม `line_net_amount` ของ `fact_sales_line` = 23,175,222.85
2. ผลรวม `payment_amount` ของ `fact_payment` = ผลรวม `receipt_total` ของแต่ละใบเสร็จ

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **จุดที่นักศึกษามักพลาด** ผลรวมของ `line_net_amount` **ไม่เท่ากับ** ผลรวมของ `receipt_total`
> เพราะมี **ส่วนลดท้ายบิล** ที่ผูกกับใบเสร็จ ไม่ได้ผูกกับรายการสินค้า
>
> ในการออกแบบจริงต้องตัดสินใจว่าจะ *ปันส่วน* ส่วนลดท้ายบิลลงแต่ละรายการหรือไม่
> ซึ่งเป็นการตัดสินใจเชิงธุรกิจ ไม่ใช่เชิงเทคนิค

## ส่วนที่ 5 — เขียน Grain Contract

### 🧑‍💻 งานที่ 5 (เขียนเป็นข้อความ ไม่ต้องเขียนโค้ด)

เติมประโยคต่อไปนี้ให้สมบูรณ์สำหรับ fact table ทั้งสองตาราง

> หนึ่งแถวใน `fact_sales_line` แทน __________ ของ __________ ณ __________
>
> หนึ่งแถวใน `fact_payment` แทน __________ ของ __________ ณ __________

จากนั้นตอบคำถาม
1. ถ้าอนาคตธุรกิจเริ่มขายแบบ "ซื้อ 1 แถม 1" grain เดิมยังใช้ได้หรือไม่ เพราะเหตุใด
2. ถ้าผู้บริหารถามว่า "ลูกค้าที่จ่ายด้วยพร้อมเพย์ซื้อสินค้าหมวดใดมากที่สุด"
   ต้องใช้ตารางใดบ้าง และมีความเสี่ยงอะไรในการ join
3. เขียนกฎทดสอบ (assertion) อย่างน้อย 3 ข้อ ที่ควรรันทุกครั้งหลังโหลดข้อมูล
   เพื่อยืนยันว่า grain ยังถูกต้อง

In [ ]:
# เขียนโค้ดของคุณที่นี่


---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — คำนวณและอธิบายการนับซ้ำได้ถูกต้อง | 2 |
| งานที่ 2 — ระบุต้นตอของการนับซ้ำได้ | 2 |
| งานที่ 3 — ตารางเปรียบเทียบครบ 4 grain × 6 คำถาม | 3 |
| งานที่ 4 — สร้าง fact table 2 ตารางและกระทบยอดผ่าน | 2 |
| งานที่ 5 — grain contract และคำตอบเชิงวิเคราะห์ | 3 |
| **รวม** | **12** |

> 💡 ตัวเลขทุกตัวในสมุดเล่มนี้ต้องตรงกับที่แสดงบนสื่อจำลอง `/sims/grain-detective`
> ถ้าไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด — ให้ย้อนกลับไปตรวจก่อนส่ง